* Baseline 2: – Sentence Embedding + Cosine
- Dùng mô hình embedding phổ biến
- Bắt được ngữ nghĩa
- Chạy ổn định, có kết quả
- So sánh được với TF-IDF

In [2]:
import pandas as pd
import numpy as np
import re
import os
from pathlib import Path

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Cell 2: Load data + clean text + create job_texts

In [ ]:
from pathlib import Path
import pandas as pd
import re

# --- Paths (portable) ---
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "datasets"
OUT_DIR = BASE_DIR / "outputs" / "jobrec_emb"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ===== Choose dataset =====
# Option A: dùng path tuyệt đối (như bạn đang làm)
jobs_path = Path("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/datasets/jobs_merged_it.xlsx")
cvs_path  = Path("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/datasets/UpdatedResumeDataSet.csv")


jobs = pd.read_excel(jobs_path)
resumes = pd.read_csv(cvs_path)
# Create a stable job id if dataset doesn't have one
if "id" not in jobs.columns:
    jobs = jobs.reset_index(drop=True)
    jobs["id"] = jobs.index.astype(int)

print("Jobs shape:", jobs.shape)
print("Resumes shape:", resumes.shape)
print("Jobs columns:", list(jobs.columns))
print("Resumes columns:", list(resumes.columns))

# --- Basic cleaning ---
# đưa về lowercase, bỏ ký tự đặc biệt, bỏ xuống dòng, gom nhiều khoảng trắng
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = text.replace("\n", " ").replace("\r", " ")
    text = re.sub(r"[^a-z0-9\s\+\#\.]", " ", text)  # keep + # . if you want (c++, c#, node.js)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# ===== Schema adapter for merged jobs =====
# jobs_merged_it.xlsx has: title, description, company, location, source
# old it_jobs.xlsx had: cleaned_description
if "cleaned_description" not in jobs.columns:
    if "description" in jobs.columns:
        jobs["cleaned_description"] = jobs["description"]
    elif "job_description" in jobs.columns:
        jobs["cleaned_description"] = jobs["job_description"]
    else:
        raise ValueError("jobs missing description column to create cleaned_description")

# Keep metadata columns clean (so output table looks clean)
jobs["title"] = jobs.get("title", "").fillna("").astype(str).str.strip()
jobs["company"] = jobs.get("company", "Unknown").fillna("Unknown").astype(str).str.strip()
jobs["location"] = jobs.get("location", "").fillna("").astype(str).str.strip()

# Create job_texts: title + cleaned_description
job_texts = (jobs["title"] + " " + jobs["cleaned_description"].fillna("").astype(str)).apply(clean_text)

# Create cv_texts (UpdatedResumeDataSet.csv uses Resume)
if "Resume" not in resumes.columns:
    raise ValueError("resumes dataset missing required column: Resume")

cv_texts = resumes["Resume"].fillna("").astype(str).apply(clean_text)

print("job_texts:", job_texts.shape, "cv_texts:", cv_texts.shape)


Jobs shape: (89277, 6)
Resumes shape: (962, 2)
Jobs columns: ['title', 'description', 'company', 'location', 'source', 'id']
Resumes columns: ['Category', 'Resume']
job_texts: (89277,) cv_texts: (962,)


## Cell 3 (load model)

In [4]:
# Tạo model embedding

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
# tên mô hình embeđing được train sẵn dùng để biến văn bản thành vector số
model = SentenceTransformer(MODEL_NAME)


## Cell 4 (encode job embeddings)
- Để KHÔNG phải encode lại job embeddings mỗi lần chạy notebook

In [5]:
# Chuỗi này có dấu / → không dùng được làm tên file; thay mọi thứ lạ bằng _ để tạo tên file an toàn
safe_model_name = re.sub(r"[^a-zA-Z0-9_-]+", "_", MODEL_NAME)

emb_path = OUT_DIR / f"job_embeddings_{safe_model_name}.npy" # đường dẫn lưu emb

if emb_path.exists():
    job_embeddings = np.load(emb_path)
else:
    # Biến toàn bộ job_texts thành vector:
    job_embeddings = model.encode(
        job_texts.tolist(),     # Danh sách text của tất cả job, mỗi job = 1 chuỗi
        batch_size=64,          # endcode 64 lần(phụ thuộc vào RAM máy; số lượng job và model_size)
        show_progress_bar=True, # thanh tiến trình 
        normalize_embeddings=True # giúp cosine ổn định hơn (vector chuẩn hoá) = 1
    )
    np.save(emb_path, job_embeddings)

job_embeddings.shape


(21961, 384)

## Cell 5 (hàm recommend)

In [6]:
def recommend_jobs_from_cv_embedding(cv_index: int, top_k: int = 10, dedupe_title_company: bool = False):
    cv_text = clean_text(resumes.loc[cv_index, "Resume"]) # Lấy CV text
    cv_embedding = model.encode([cv_text], normalize_embeddings=True)

    scores = cosine_similarity(cv_embedding, job_embeddings).flatten() # so sánh vector của CV và Job
    top_idx = np.argsort(scores)[::-1][: max(top_k * 3, top_k)]  # sắp xếp score từ cao - thấp

    results = jobs.loc[top_idx, ["id", "title", "company", "location"]].copy()
    results["score"] = scores[top_idx] # thêm cột score để tạo ra bảng kết quả

    # bỏ job trùng nhau và chỉ giữ lại 1
    if dedupe_title_company:
        results["_key"] = (results["title"].str.lower().str.strip() + "||" + results["company"].str.lower().str.strip())
        results = results.drop_duplicates("_key").drop(columns=["_key"])

    return results.head(top_k).reset_index(drop=True) # trả về topK cuối cùng


In [9]:
recommend_jobs_from_cv_embedding(cv_index=0, top_k=20)
 # score càng cao thì càng gợi ý đúng

,id,title,company,location,score
0,14341,IT Deskside Support,Infinite Computing Systems,"Cedar Rapids, IA, US",0.581874
1,10579,Kronos Consultant,ASM Tech Solutions,"Remote, US",0.553800
2,18152,Information Technology Specialist,University of Hawaii system,"Honolulu, HI, US",0.546105
3,8317,Payor Network Coordinator - Onsite/hybrid role,"Edwards Health Care Services, Inc.","Hudson, OH, US",0.544319
4,8951,Data Center Security Manager (Individual Contr...,Amazon Web Services,"Sterling, VA, US",0.539957
5,19067,Vice President; Software Engineer,Bank of America,"Jersey City, NJ, US",0.538987
6,2371,Machine Learning Research Engineer,Integrity Careers LLC,"San Francisco, CA, USA",0.532944
7,684,Scientific Computer Programmer,"SHB Technology, Inc","Bethesda, MD, US",0.529824
8,21455,Salesforce Developer with CPQ Expertise,Concentrix,"Remote, US",0.528502
9,20736,Systems Analyst,Dutech Systems Inc,"Austin, TX, US",0.526691


### Diễn giải Embedding Similarity Score (Cosine Similarity)

| Khoảng điểm | Diễn giải |
|------------|-----------|
| < 0.30 | Gần như không liên quan về ngữ nghĩa |
| 0.30 – 0.45 | Liên quan yếu |
| 0.45 – 0.55 | Liên quan ở mức trung bình |
| **0.55 – 0.65** | **Liên quan tốt** |
| **0.65 – 0.75** | **Liên quan rất tốt** |
| > 0.75 | Liên quan cực mạnh (hiếm gặp) |


In [10]:
# Save a few demo results (so dễ bỏ vào báo cáo)
demo_indices = [0, 5, 20, 100]

for cv_index in demo_indices:
    results = recommend_jobs_from_cv_embedding(cv_index=cv_index, top_k=10)
    results.to_csv(OUT_DIR / f"baseline_cv_{cv_index}.csv", index=False)

print("Saved to:", OUT_DIR)


Saved to: outputs/jobrec_emb
